In [5]:
import os
from os import makedirs
from os.path import exists

import numpy as np
import plotly.graph_objs as go

import h5py

from scipy.signal import sosfiltfilt, butter, zpk2sos, detrend

In [6]:
amp = np.random.randn(3,4)

In [10]:
1/23

0.043478260869565216

In [2]:
# 声明全局变量
weightData = None

In [3]:
def get_msg(file_path):
    folder = os.path.dirname(file_path)
    full_filename = os.path.basename(file_path)
    filename, _ = os.path.splitext(full_filename)

    dev_id = str.split(filename, '-')[0]
    date = str.split(filename, '-')[1][0:8]
    hour = str.split(filename, '-')[1][8:]

    msg = {
        'folder': folder,
        'full_filename': full_filename,
        'filename': filename,
        'dev_id': dev_id,
        'date': date,
        'hour': hour
    }

    return msg

In [4]:
def func_cancel(inputData, i_frame):
    global weightData
    weight = 0.95

    if i_frame == 0:
        weightData = inputData
        diffData = np.ones_like(inputData)
    else:
        if i_frame == 1:
            diffData = inputData - weightData
            weightData = 0.5 * inputData + 0.5 * weightData
        else:
            diffData = inputData - weightData
            weightData = (1 - weight) * inputData + weight * weightData

    diffData = np.abs(diffData)

    return diffData

In [5]:
def butter_bandpass(lowcut, highcut, fs=20, order=8):
    z, p, k = butter(order, [lowcut, highcut], btype='bandpass', fs=fs, output='zpk')
    sos = zpk2sos(z, p, k)
    return butter(order, [lowcut, highcut], btype='bandpass', fs=fs, output='sos')

In [6]:
sos = butter_bandpass(0.1, 2, 20, order=8)

In [7]:
def get_merged_data_hdf5(hdf5_file):

    rows_minutes = 60
    n_cols = 23
    merged_data = np.full([rows_minutes * 1200, n_cols], np.nan).astype(np.complex64)

    with h5py.File(hdf5_file, 'r') as file:
        cur_data = file['data'][:]
        frame_ids = file['frame_ids'][:]
        # print(frame_ids)

    start_id = frame_ids[0]
    start_index = 0
    frame_len = 5 * 20
    for x, id in enumerate(frame_ids):
        if start_index >= 72000:
            break

        if x == 0:
            if not frame_ids[0]+1 == frame_ids[1]:
                if frame_ids[1]+1 == frame_ids[2]:
                    start_id = frame_ids[1]
                    continue

        if id == start_id:
            merged_data[start_index: start_index+frame_len, :] = cur_data[x*frame_len: (x+1)*frame_len, :]

            start_id = id + 1
            start_index += frame_len

        elif id > start_id:

            if x+1 == len(frame_ids):
                merged_data[start_index: start_index+frame_len, :] = cur_data[x*frame_len: (x+1)*frame_len, :]

                start_id = id + 1
                start_index += frame_len
            else:
                if frame_ids[x+1] == start_id:
                    continue
                else:
                    start_index += (id - start_id) * frame_len

                    merged_data[start_index: start_index+frame_len, :] = cur_data[x*frame_len: (x+1)*frame_len, :]

                    start_index += frame_len
                    start_id = id + 1
        
        elif id < start_id:
            if x+1 == len(frame_ids):
                merged_data[start_index: start_index+frame_len, :] = cur_data[x*frame_len: (x+1)*frame_len, :]

                start_id = id + 1
                start_index += frame_len
            else:
                if frame_ids[x+1] == start_id:
                    continue
                else:
                    merged_data[start_index: start_index+frame_len, :] = cur_data[x*frame_len: (x+1)*frame_len, :]

                    start_id = id + 1
                    start_index += frame_len

    # 如果丢太多，就用老版本的代码
    lost_frame = np.sum(np.isnan(merged_data)) // 2300
    all_frame = len(frame_ids)
    if lost_frame / all_frame >= 0.1:
        return get_merged_data_hdf5_old(hdf5_file)
    else:
        # 使用前一个有效值填充缺失值
        merged_data = fill_missing_with_zero_vectorized(merged_data)
        return merged_data, lost_frame


def get_merged_data_hdf5_old(hdf5_file_path):

    rows_minutes = 60
    n_cols = 23
    merged_data = np.full([rows_minutes * 1200, n_cols], np.nan).astype(np.complex64)
    lost_frame = 0

    with h5py.File(hdf5_file_path, 'r') as file:
        cur_data = file['data'][:]

    if len(cur_data) >= rows_minutes * 1200:
        merged_data[:, :] = cur_data[0:rows_minutes * 1200, :]
    else:
        merged_data[0:len(cur_data), :] = cur_data
        lost_frame = rows_minutes * 1200 - len(cur_data)

    merged_data = fill_missing_with_zero_vectorized(merged_data)

    return merged_data, lost_frame


def fill_missing_with_zero_vectorized(arr):
    # 创建结果数组副本
    result = arr.copy()
    
    # 按列处理
    for col in range(arr.shape[1]):
        column_data = arr[:, col]
        mask = np.isnan(column_data)
        
        # 找到非NaN值的索引
        valid_indices = np.where(~mask)[0]
        
        if len(valid_indices) == 0:
            # 如果整列都是NaN，全部填充0
            result[:, col] = 0
            continue
        
        # 构建索引数组
        idx = np.arange(len(column_data))
        # 对于每个位置，找到前一个有效值的索引
        idx = np.maximum.accumulate(np.where(~mask, idx, 0))
        
        # 获取填充值
        filled_values = column_data[idx]
        
        # 处理开头的NaN（idx=0的位置且原始值为NaN）
        filled_values[(idx == 0) & mask] = 0 + 0j
        
        result[:, col] = filled_values
    
    return result

In [8]:
def draw_all_channel(hdf5_file, html_save_path):
    
    msg = get_msg(hdf5_file)
    dev_id = msg['dev_id']
    date = msg['date']
    hour = msg['hour']

    # 创建图表
    fig = go.Figure()

    merged_data, _ = get_merged_data_hdf5(hdf5_file)

    # 获取幅值
    diffData = np.zeros_like(merged_data, dtype=float)
    # 获取相位
    cur_phase = np.unwrap(np.angle(merged_data[:, 0:23]).T).T

    # ------------------------------------------------------------------------------------------------------------------
    # 创建图表
    fig = go.Figure()
    for i in range(merged_data.shape[0]):
        diffData[i, :] = func_cancel(merged_data[i, :], i)

    for i in range(diffData.shape[1]):
        x = [x/1200 for x in range(diffData.shape[0])]
        y = diffData[:, i].T
        fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name=f'{i+1}'))

    # 保存为 HTML 文件
    if not exists(html_save_path):
        makedirs(html_save_path)
    fig.write_html(os.path.join(html_save_path, f'{dev_id}-{date}-{hour}-对消幅值.html'))

    # ------------------------------------------------------------------------------------------------------------------
    # 创建图表
    fig = go.Figure()
    for i in range(merged_data.shape[0]):
        diffData[i, :] = func_cancel(merged_data[i, :], i)

    for i in range(diffData.shape[1]):

        x = [x/1200 for x in range(diffData.shape[0])]
        y = np.abs(merged_data[:, i].T)
        
        fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name=f'{i+1}'))

    # 保存为 HTML 文件
    if not exists(html_save_path):
        makedirs(html_save_path)
    fig.write_html(os.path.join(html_save_path, f'{dev_id}-{date}-{hour}-原始幅值.html'))

    # ------------------------------------------------------------------------------------------------------------------
    # 创建图表
    fig = go.Figure()

    # 获取相位
    cur_phase = np.unwrap(np.angle(merged_data[:, 0:23]).T).T
    print(cur_phase.shape)
    del merged_data

    for i in range(cur_phase.shape[1]):

        x = [x/1200 for x in range(cur_phase.shape[0])]
        y = sosfiltfilt(sos, cur_phase[:, i].T)
        fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name=f'{i+1}'))

    # 设置y轴范围
    fig.update_layout(
        yaxis=dict(
            range=[-20, 20]
        )
    )

    # 保存为 HTML 文件
    if not exists(html_save_path):
        makedirs(html_save_path)
    fig.write_html(os.path.join(html_save_path, f'{dev_id}-{date}-{hour}-相位.html'))

In [ ]:
html_save_path = r'E:\5.毫米波雷达\code\呼吸率\dataset\倾斜\html'
hdf5_folder = r'E:\5.毫米波雷达\code\呼吸率\dataset\倾斜\hdf5'
hdf5_list = os.listdir(hdf5_folder)

for hdf5 in hdf5_list:
    hdf5_path = os.path.join(hdf5_folder, hdf5)

    draw_all_channel(hdf5_path, html_save_path)

In [12]:
hdf5_folder = r'E:\5.毫米波雷达\code\活死猪\20251016'
html_save_path = r'E:\5.毫米波雷达\code\活死猪\html\20251016'
hdf5_list = os.listdir(hdf5_folder)

for hdf5 in hdf5_list:
    # if '30006' not in hdf5:
    #     continue
    hdf5_path = os.path.join(hdf5_folder, hdf5)

    draw_all_channel(hdf5_path, html_save_path)

(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
(72000, 23)
